<a href="https://colab.research.google.com/github/Vysokodelovoi/IAD_GP/blob/main/gp_ml.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Построение моделей машинного обучения до FE. Классификация.

In [2]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 6.7 MB/s eta 0:00:00


In [3]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.3 MB/s eta 0:00:00


In [33]:
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.model_selection import cross_val_score, StratifiedKFold, train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler, OneHotEncoder
from sklearn.decomposition import PCA
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score
import catboost as cb
from catboost import CatBoostClassifier
import optuna
import mlutils as mu

In [5]:
df = pd.read_parquet('main_df_edited_no_dup.parquet')

In [6]:
df

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date,country_full,arrival_date_month_num
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,0,Transient,0.00,0,0,Check-Out,2015-07-01,Portugal,6
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,0,Transient,0.00,0,0,Check-Out,2015-07-01,Portugal,6
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,0,Transient,75.00,0,0,Check-Out,2015-07-02,United Kingdom,6
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,0,Transient,75.00,0,0,Check-Out,2015-07-02,United Kingdom,6
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,0,Transient,98.00,0,1,Check-Out,2015-07-03,United Kingdom,6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
118893,City Hotel,0,23,2017,August,35,30,2,5,2,...,No Deposit,0,Transient,96.14,0,0,Check-Out,2017-09-06,Belgium,7
118894,City Hotel,0,102,2017,August,35,31,2,5,3,...,No Deposit,0,Transient,225.43,0,2,Check-Out,2017-09-07,France,7
118895,City Hotel,0,34,2017,August,35,31,2,5,2,...,No Deposit,0,Transient,157.71,0,4,Check-Out,2017-09-07,Germany,7
118896,City Hotel,0,109,2017,August,35,31,2,5,2,...,No Deposit,0,Transient,104.40,0,0,Check-Out,2017-09-07,United Kingdom,7


In [7]:
# Линейные модели logistic_regression / linear, Даниэль
#  svm Игорь
#  boosting Игорь
#  random trees Даниэль
#  нейронка Игорь
#  изотоническая регрессия Даниэль
#  ранжирование попробовать? Игорь

#  Для каждой затюнить гиперпараметры через GridSearchCV или optuna
#  Результаты по проведенным экспериментам собрать в pandas датасет и отправить
#  Сделать выводы по performance моделю, посмотреть на важность признаков через shap или feature_importances


#  Графики с дубликатами и без сравнить

In [8]:
df_model = df[[c for c in df.columns if c not in ['reservation_status', 'reservation_status_date', 'country_full']]]

In [9]:
df_model

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,reserved_room_type,assigned_room_type,booking_changes,deposit_type,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,arrival_date_month_num
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,C,C,3,No Deposit,0,Transient,0.00,0,0,6
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,C,C,4,No Deposit,0,Transient,0.00,0,0,6
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,A,C,0,No Deposit,0,Transient,75.00,0,0,6
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,A,A,0,No Deposit,0,Transient,75.00,0,0,6
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,A,A,0,No Deposit,0,Transient,98.00,0,1,6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
118893,City Hotel,0,23,2017,August,35,30,2,5,2,...,A,A,0,No Deposit,0,Transient,96.14,0,0,7
118894,City Hotel,0,102,2017,August,35,31,2,5,3,...,E,E,0,No Deposit,0,Transient,225.43,0,2,7
118895,City Hotel,0,34,2017,August,35,31,2,5,2,...,D,D,0,No Deposit,0,Transient,157.71,0,4,7
118896,City Hotel,0,109,2017,August,35,31,2,5,2,...,A,A,0,No Deposit,0,Transient,104.40,0,0,7


In [10]:
X = df_model[[c for c in df_model.columns if c != 'is_canceled']].copy()
y = df_model.is_canceled.copy()

In [11]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=8)

## SVM

In [ ]:
num_features = [
    'lead_time', 'arrival_date_year', 'arrival_date_week_number',
    'arrival_date_day_of_month', 'stays_in_weekend_nights',
    'stays_in_week_nights', 'adults', 'children', 'babies',
    'is_repeated_guest', 'previous_cancellations',
    'previous_bookings_not_canceled', 'booking_changes',
    'days_in_waiting_list', 'required_car_parking_spaces',
    'total_of_special_requests'
]

cat_features = [
    'hotel', 'arrival_date_month', 'meal', 'country',
    'market_segment', 'distribution_channel', 'reserved_room_type',
    'assigned_room_type', 'deposit_type', 'customer_type'
]

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('scaler', StandardScaler(), num_features),
        ('cat', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), cat_features)
    ]
)

In [ ]:
svm_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('pca', PCA()),
    ('classifier', LinearSVC(dual=False, random_state=8)),
])

In [ ]:
def objective(trial):

    max_iter_param = trial.suggest_int('classifier__max_iter', 1000, 10000, step=1000)
    tol_param = trial.suggest_float('classifier__tol', 1e-5, 1e-2, log=True)
    c_param = trial.suggest_float('classifier__C', 1e-4, 1e2, log=True)

    scaler_param = trial.suggest_categorical('scaler_type', ['standard', 'robust'])
    scaler = StandardScaler() if scaler_param == 'standard' else RobustScaler()

    use_pca = trial.suggest_categorical('use_pca', [True, False])

    if use_pca:
        pca_components = trial.suggest_int('pca__n_components', 1, 35)
        pca_step = PCA(n_components=pca_components)
    else:
        pca_step = 'passthrough'

    svm_pipeline.set_params(
        preprocessor__scaler=scaler,
        pca=pca_step,
        classifier__C=c_param,
        classifier__max_iter=max_iter_param,
        classifier__tol=tol_param
    )

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=8)
    scores = cross_val_score(
        svm_pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring='f1',
        n_jobs=-1,
        error_score='raise'
    )
    return np.mean(scores)

In [ ]:
# study = optuna.create_study(direction='maximize')
# study.optimize(objective, show_progress_bar=True, n_trials=50)

# print("Лучшие параметры:", study.best_params)
# print("Лучший F1-score:", study.best_value)

In [ ]:
# best = study.best_params

In [ ]:
best = {'classifier__max_iter': 10000, 'classifier__tol': 0.0008743508260033666, 'classifier__C': 36.856060983954215, 'scaler_type': 'standard', 'use_pca': False}

best_scaler = StandardScaler() if best['scaler_type'] == 'standard' else RobustScaler()

if best.get('use_pca', False):
    best_pca = PCA(n_components=best['pca__n_components'])
else:
    best_pca = 'passthrough'

svm_pipeline.set_params(
    preprocessor__scaler=best_scaler,
    pca=best_pca,
    classifier__C=best['classifier__C'],
    classifier__max_iter=best['classifier__max_iter'],
    classifier__tol=best['classifier__tol']
)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('scaler', StandardScaler(),
                                                  ['lead_time',
                                                   'arrival_date_year',
                                                   'arrival_date_week_number',
                                                   'arrival_date_day_of_month',
                                                   'stays_in_weekend_nights',
                                                   'stays_in_week_nights',
                                                   'adults', 'children',
                                                   'babies',
                                                   'is_repeated_guest',
                                                   'previous_cancellations',
                                                   'previous_bookings_not_canceled',
                                                   'booking_change...
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  ['hotel',
                                                   'arrival_date_month', 'meal',
                                                   'country', 'market_segment',
                                                   'distribution_channel',
                                                   'reserved_room_type',
                                                   'assigned_room_type',
                                                   'deposit_type',
                                                   'customer_type'])])),
                ('pca', 'passthrough'),
                ('classifier',
                 LinearSVC(C=36.856060983954215, dual=False, max_iter=10000,
                           random_state=8, tol=0.0008743508260033666))])

In [ ]:
import importlib
import ml_utils as mu

# Эта команда принудительно обновит модуль в памяти Jupyter
importlib.reload(mu)

<module 'ml_utils' from '/content/ml_utils.py'>

In [ ]:
mu.run_experiment(svm_pipeline, 'svm', 'main_df_edited_no_dup.parquet', best)

svm | score: 0.55317 time: 5.7s


/content/ml_utils.py:50: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results_df = pd.concat([results_df, pd.DataFrame([result_row])], ignore_index=True)


{'timestamp': '2026-06-14T22:58:17.660727',
 'method': 'svm',
 'params': '{"classifier__max_iter": 10000, "classifier__tol": 0.0008743508260033666, "classifier__C": 36.856060983954215, "scaler_type": "standard", "use_pca": false}',
 'score': 0.553168880455408,
 'duration_sec': 5.713756799697876}

## CatBoost

In [ ]:
X = df_model[[c for c in df_model.columns if c not in ['is_canceled','country_full']]].copy()
y = df_model.is_canceled.copy()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=8)

In [ ]:
num_features = [
    'lead_time', 'arrival_date_year', 'arrival_date_week_number',
    'arrival_date_day_of_month', 'stays_in_weekend_nights',
    'stays_in_week_nights', 'adults', 'children', 'babies',
    'is_repeated_guest', 'previous_cancellations',
    'previous_bookings_not_canceled', 'booking_changes',
    'days_in_waiting_list', 'required_car_parking_spaces',
    'total_of_special_requests'
]

cat_features = [
    'hotel', 'arrival_date_month', 'meal', 'country',
    'market_segment', 'distribution_channel', 'reserved_room_type',
    'assigned_room_type', 'deposit_type', 'customer_type'
]

In [ ]:
def objective(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 200, 1000, step=100),
        'depth': trial.suggest_int('depth', 4, 6),
        'learning_rate': trial.suggest_float('learning_rate', 0.03, 0.2, log=True),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-2, 10.0, log=True),
        'cat_features': cat_features,
        'random_seed': 8,
        'thread_count': -1,
        'early_stopping_rounds': trial.suggest_int('early_stopping_rounds', 10, 100, step=5)
    }

    model = CatBoostClassifier(**params)

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=8)
    scores = []
    for train_idx, val_idx in cv.split(X_train, y_train):

        X_train_train = X_train.iloc[train_idx]
        X_train_val = X_train.iloc[val_idx]
        y_train_train = y_train.iloc[train_idx]
        y_train_val = y_train.iloc[val_idx]

        model = cb.CatBoostClassifier(**params)

        model.fit(
            X_train_train, y_train_train,
            eval_set=(X_train_val, y_train_val),
            verbose=False
        )

        y_pred = model.predict(X_train_val)
        scores.append(f1_score(y_train_val, y_pred))

    return np.mean(scores)

In [ ]:
study_cb = optuna.create_study(direction='maximize')
study_cb.optimize(objective, n_trials=50, show_progress_bar=True)

[I 2026-06-14 16:30:45,557] A new study created in memory with name: no-name-78b32a70-b284-4478-9fc9-28277e4a28d8


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-06-14 16:38:29,971] Trial 0 finished with value: 0.6886515832289016 and parameters: {'iterations': 1000, 'depth': 4, 'learning_rate': 0.05525117752762227, 'l2_leaf_reg': 0.9709957380220429, 'early_stopping_rounds': 85}. Best is trial 0 with value: 0.6886515832289016.
[I 2026-06-14 16:43:44,086] Trial 1 finished with value: 0.6787954117764385 and parameters: {'iterations': 700, 'depth': 4, 'learning_rate': 0.03656176676313291, 'l2_leaf_reg': 0.05305348598636522, 'early_stopping_rounds': 80}. Best is trial 0 with value: 0.6886515832289016.
[I 2026-06-14 16:50:41,822] Trial 2 finished with value: 0.6980778170816262 and parameters: {'iterations': 900, 'depth': 4, 'learning_rate': 0.15179953122208972, 'l2_leaf_reg': 0.02629742319435691, 'early_stopping_rounds': 85}. Best is trial 2 with value: 0.6980778170816262.
[I 2026-06-14 16:56:52,358] Trial 3 finished with value: 0.6930103966573767 and parameters: {'iterations': 1000, 'depth': 5, 'learning_rate': 0.06900829323089025, 'l2_leaf_

KeyboardInterrupt: 

In [ ]:
best = {'iterations': 800, 'depth': 5, 'learning_rate': 0.1398336717868352, 'l2_leaf_reg': 0.2265442520413868, 'early_stopping_rounds': 100, 'cat_features': cat_features}

In [ ]:
import mlutils as mu

In [ ]:
cb_pipeline = CatBoostClassifier(**best)
mu.run_experiment(cb_pipeline, 'catboost', 'main_df_edited_no_dup.parquet', param_dict=best)

0:	learn: 0.6116590	total: 145ms	remaining: 1m 56s
1:	learn: 0.5702473	total: 254ms	remaining: 1m 41s
2:	learn: 0.5412556	total: 337ms	remaining: 1m 29s
3:	learn: 0.5123410	total: 418ms	remaining: 1m 23s
4:	learn: 0.4885666	total: 476ms	remaining: 1m 15s
5:	learn: 0.4716983	total: 535ms	remaining: 1m 10s
6:	learn: 0.4575741	total: 602ms	remaining: 1m 8s
7:	learn: 0.4466495	total: 666ms	remaining: 1m 5s
8:	learn: 0.4410961	total: 745ms	remaining: 1m 5s
9:	learn: 0.4356693	total: 823ms	remaining: 1m 5s
10:	learn: 0.4301783	total: 904ms	remaining: 1m 4s
11:	learn: 0.4266186	total: 982ms	remaining: 1m 4s
12:	learn: 0.4196575	total: 1.06s	remaining: 1m 4s
13:	learn: 0.4156020	total: 1.13s	remaining: 1m 3s
14:	learn: 0.4128498	total: 1.21s	remaining: 1m 3s
15:	learn: 0.4097617	total: 1.28s	remaining: 1m 2s
16:	learn: 0.4080068	total: 1.36s	remaining: 1m 2s
17:	learn: 0.4062892	total: 1.44s	remaining: 1m 2s
18:	learn: 0.4051317	total: 1.5s	remaining: 1m 1s
19:	learn: 0.4018052	total: 1.59s	re

/content/mlutils.py:50: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results_df = pd.concat([results_df, pd.DataFrame([result_row])], ignore_index=True)


{'timestamp': '2026-06-15T18:35:35.296278',
 'method': 'catboost',
 'params': '{"iterations": 800, "depth": 5, "learning_rate": 0.1398336717868352, "l2_leaf_reg": 0.2265442520413868, "early_stopping_rounds": 100, "cat_features": ["hotel", "arrival_date_month", "meal", "country", "market_segment", "distribution_channel", "reserved_room_type", "assigned_room_type", "deposit_type", "customer_type"]}',
 'score': 0.7067760150578113,
 'duration_sec': 65.63863396644592}

## TabNet

In [ ]:
!pip install pytorch-tabnet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.5/44.5 kB 2.4 MB/s eta 0:00:00


In [ ]:
from pytorch_tabnet.tab_model import TabNetClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OrdinalEncoder
import torch

In [ ]:
X = df_model[[c for c in df_model.columns if c not in ['is_canceled','country_full']]].copy()
y = df_model.is_canceled.copy()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=8)

In [ ]:
num_features = [
    'lead_time', 'arrival_date_year', 'arrival_date_week_number',
    'arrival_date_day_of_month', 'stays_in_weekend_nights',
    'stays_in_week_nights', 'adults', 'children', 'babies',
    'is_repeated_guest', 'previous_cancellations',
    'previous_bookings_not_canceled', 'booking_changes',
    'days_in_waiting_list', 'required_car_parking_spaces',
    'total_of_special_requests'
]

cat_features = [
    'hotel', 'arrival_date_month', 'meal', 'country',
    'market_segment', 'distribution_channel', 'reserved_room_type',
    'assigned_room_type', 'deposit_type', 'customer_type'
]

In [ ]:
categories_list = [list(X_train[col].astype(str).unique()) for col in cat_features]

In [ ]:
cat_dims = [int(X_train[col].nunique()) for col in cat_features]
cat_idxs = list(range(len(cat_features)))

n_d (Dimension of prediction layer): Размерность слоя предсказаний. Чем больше датасет и чем больше в нем сложных взаимосвязей, тем выше должно быть это число (обычно от 8 до 64).

n_a (Dimension of attention layer): Размерность слоя внимания.

Золотое правило TabNet: Практически всегда выставляют n_d = n_a. Чтобы не тратить время Optuna на перебор бессмысленных комбинаций, лучше заставить её подбирать одно число для обоих параметров.

n_steps: Количество шагов архитектуры (можно метафорически сравнить с количеством последовательных деревьев в бустинге). Обычно подбирается в диапазоне от 3 до 10. Больше 10 часто ведет к дикому переобучению и долгому обучению.

gamma: Коэффициент повторного использования признаков на разных шагах внимания. Если gamma = 1.0, признаки выбираются независимо на каждом шаге. Диапазон: от 1.0 до 2.0.

n_independent и n_shared: Количество независимых и разделяемых (shared) блоков GLU (Gated Linear Units). Обычно берут от 1 до 5.

Регуляризация и обучение
lambda_sparse: Пожалуй, самый критичный параметр. Он отвечает за "разреженность" (sparsity) внимания - заставлять ли модель выбирать только самые важные признаки или размазывать внимание по всем. Варьируется логарифмически от 1e-6 до 1e-1.

mask_type: Тип функции активации для выбора признаков: 'sparsemax' (жестко зануляет неважные признаки) или 'entmax' (более мягкий вариант).

virtual_batch_size: Размер батча для Ghost Batch Normalization. TabNet очень чувствителен к нормализации. Обычно основной batch_size делают большим (например, 1024 или 2048), а virtual_batch_size — поменьше (32, 64, 128). Он обязательно должен быть делителем основного батча!

In [ ]:
def objective_tabnet(trial):
    n_da = trial.suggest_int('n_da', 8, 64, step=8)
    batch_size = trial.suggest_categorical('batch_size', [512, 1024, 2048])
    virtual_batch_size = trial.suggest_categorical('virtual_batch_size', [32, 64, 128])

    if virtual_batch_size > batch_size:
        virtual_batch_size = batch_size

    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=8)
    scores = []

    for train_idx, val_idx in cv.split(X_train, y_train):
        X_tr, X_va = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr = y_train.iloc[train_idx] if hasattr(y_train, 'iloc') else y_train[train_idx]
        y_va = y_train.iloc[val_idx] if hasattr(y_train, 'iloc') else y_train[val_idx]

        preprocessor = ColumnTransformer(
            transformers=[
                ('cat', OrdinalEncoder(
                    categories=categories_list,
                    handle_unknown='use_encoded_value',
                    unknown_value=-1
                ), cat_features)
            ],
            remainder='passthrough'
        )

        classifier = TabNetClassifier(
            cat_idxs=cat_idxs,
            cat_dims=cat_dims,
            n_d=n_da,
            n_a=n_da,
            n_steps=trial.suggest_int('n_steps', 3, 8),
            gamma=trial.suggest_float('gamma', 1.0, 1.8),
            lambda_sparse=trial.suggest_float('lambda_sparse', 1e-5, 1e-1, log=True),
            n_independent=trial.suggest_int('n_independent', 1, 4),
            n_shared=trial.suggest_int('n_shared', 1, 4),
            mask_type=trial.suggest_categorical('mask_type', ['sparsemax', 'entmax']),
            optimizer_fn=torch.optim.Adam,
            optimizer_params=dict(lr=trial.suggest_float('lr', 1e-3, 5e-2, log=True)),
            scheduler_fn=torch.optim.lr_scheduler.StepLR,
            scheduler_params={"step_size": 10, "gamma": 0.9},
            verbose=0,
            device_name='cuda' if torch.cuda.is_available() else 'cpu'
        )

        tabnet_pipeline = Pipeline([
            ('preprocessor', preprocessor),
            ('classifier', classifier)
        ])

        preprocessor.fit(X_tr)
        X_va_transformed = preprocessor.transform(X_va)

        y_tr_arr = np.asarray(y_tr)
        y_va_arr = np.asarray(y_va)

        tabnet_pipeline.fit(
            X_tr, y_tr_arr,
            classifier__eval_set=[(X_va_transformed, y_va_arr)],
            classifier__eval_metric=['balanced_accuracy'],
            classifier__max_epochs=50,
            classifier__patience=10,
            classifier__batch_size=batch_size,
            classifier__virtual_batch_size=virtual_batch_size
        )

        preds = tabnet_pipeline.predict(X_va)
        scores.append(f1_score(y_va_arr, preds))

    return np.mean(scores)

In [ ]:
study_tabnet = optuna.create_study(direction='maximize')
study_tabnet.optimize(objective_tabnet, n_trials=20, show_progress_bar=True)

[I 2026-06-15 20:33:58,509] A new study created in memory with name: no-name-975a9ff1-0618-4269-971c-e6b2f43cd17a


  0%|          | 0/20 [00:00<?, ?it/s]


Early stopping occurred at epoch 11 with best_epoch = 1 and best_val_0_balanced_accuracy = 0.52874


/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


Stop training because you reached max_epochs = 50 with best_epoch = 40 and best_val_0_balanced_accuracy = 0.72582


...


Early stopping occurred at epoch 35 with best_epoch = 25 and best_val_0_balanced_accuracy = 0.78867


/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)



Early stopping occurred at epoch 28 with best_epoch = 18 and best_val_0_balanced_accuracy = 0.78843


/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


[I 2026-06-16 00:41:46,737] Trial 19 finished with value: 0.6366677800293881 and parameters: {'n_da': 56, 'batch_size': 1024, 'virtual_batch_size': 64, 'n_steps': 3, 'gamma': 1.3067894659429709, 'lambda_sparse': 0.003382670732225801, 'n_independent': 4, 'n_shared': 2, 'mask_type': 'entmax', 'lr': 0.0256425792146109}. Best is trial 16 with value: 0.7009890116949098.


In [ ]:
study_tabnet.best_params

{'n_da': 56,
 'batch_size': 512,
 'virtual_batch_size': 64,
 'n_steps': 3,
 'gamma': 1.2126647444379608,
 'lambda_sparse': 1.0088907457764974e-05,
 'n_independent': 4,
 'n_shared': 2,
 'mask_type': 'entmax',
 'lr': 0.04973945735856026}

In [14]:
!pip install torch

In [15]:
!pip install pytorch-tabnet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.5/44.5 kB 1.7 MB/s eta 0:00:00


In [43]:
from pytorch_tabnet.tab_model import TabNetClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OrdinalEncoder, FunctionTransformer
import torch

In [44]:
num_features = [
    'lead_time', 'arrival_date_year', 'arrival_date_week_number',
    'arrival_date_day_of_month', 'stays_in_weekend_nights',
    'stays_in_week_nights', 'adults', 'children', 'babies',
    'is_repeated_guest', 'previous_cancellations',
    'previous_bookings_not_canceled', 'booking_changes',
    'days_in_waiting_list', 'required_car_parking_spaces',
    'total_of_special_requests'
]

cat_features = [
    'hotel', 'arrival_date_month', 'meal', 'country',
    'market_segment', 'distribution_channel', 'reserved_room_type',
    'assigned_room_type', 'deposit_type', 'customer_type'
]

In [45]:
categories_list = [list(X_train[col].astype(str).unique()) for col in cat_features]

In [46]:
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', make_pipeline(
            OrdinalEncoder(
                handle_unknown='use_encoded_value',
                unknown_value=-1,
                encoded_missing_value=-1
            ),
            FunctionTransformer(lambda x: x + 1)
        ), cat_features)
    ],
    remainder='passthrough'
)

In [47]:
import scipy.sparse as sp

In [48]:
X_train_trans = preprocessor.fit_transform(X_train)

if sp.issparse(X_train_trans):
    X_train_trans_np = X_train_trans.toarray()
else:
    X_train_trans_np = np.asarray(X_train_trans)

In [49]:
X_train_trans = preprocessor.fit_transform(X_train)
X_train_trans_np = np.asarray(X_train_trans)

cat_idxs = list(range(len(cat_features)))
cat_dims = [int(X_train_trans_np[:, i].max()) + 1 for i in cat_idxs]

In [50]:
tabnet_pipeline = Pipeline([
            ('preprocessor', preprocessor),
            ('classifier', TabNetClassifier(
                cat_idxs=cat_idxs,
                cat_dims=cat_dims,
                n_d=56,
                n_a=56,
                n_steps=3,
                gamma=1.2126647444379608,
                lambda_sparse=1.0088907457764974e-05,
                n_independent=4,
                n_shared=2,
                mask_type='entmax',
                optimizer_fn=torch.optim.Adam,
                optimizer_params=dict(lr=0.04973945735856026),
                scheduler_fn=torch.optim.lr_scheduler.StepLR,
                scheduler_params={"step_size": 10, "gamma": 0.9},
                verbose=0,
                device_name = 'cpu'
        ))
        ])

In [51]:
tabnet_pipeline.fit(
    X_train, y_train.values,
    classifier__max_epochs=100,
    classifier__batch_size=512,
    classifier__virtual_batch_size=64
)

/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/abstract_model.py:687: UserWarning: No early stopping will be performed, last training weights will be used.
  warnings.warn(wrn_msg)
/usr/local/lib/python3.12/dist-packages/sklearn/compose/_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('cat',
                                                  Pipeline(steps=[('ordinalencoder',
                                                                   OrdinalEncoder(encoded_missing_value=-1,
                                                                                  handle_unknown='use_encoded_value',
                                                                                  unknown_value=-1)),
                                                                  ('functiontransformer',
                                                                   FunctionTransformer(func=<function <lambda> at 0x7907dddeb6a0>))]),
                                                  ['hotel',
                                                   'arrival_date_month', 'meal',
                                                   'co...
                                  lambda_sparse=1.0088907457764974e-05,
                                  seed=0,
                                  clip_value=1,
                                  verbose=0,
                                  optimizer_fn=<class 'torch.optim.adam.Adam'>,
                                  optimizer_params={'lr': 0.04973945735856026},
                                  scheduler_fn=<class 'torch.optim.lr_scheduler.StepLR'>,
                                  scheduler_params={'gamma': 0.9,
                                                    'step_size': 10},
                                  mask_type='entmax',
                                  input_dim=28,
                                  output_dim=2,
                                  device_name='cpu',
                                  n_shared_decoder=1,
                                  n_indep_decoder=1,
                                  grouped_features=[]))])

In [52]:
y_pred = tabnet_pipeline.predict(X_test)

In [53]:
f1_score(y_test, y_pred)

0.6686916413544831

# Построение моделей машинного обучения до FE. Регрессия.

In [ ]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 6.7 MB/s eta 0:00:00


In [ ]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.3 MB/s eta 0:00:00


In [86]:
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.model_selection import cross_val_score, StratifiedKFold, train_test_split, KFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler, OneHotEncoder
from sklearn.decomposition import PCA
from sklearn.svm import LinearSVC, LinearSVR
from sklearn.metrics import f1_score
import catboost as cb
from catboost import CatBoostClassifier
import optuna
import mlutils as mu

In [55]:
df = pd.read_parquet('main_df_edited_no_dup.parquet')

In [56]:
df

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date,country_full,arrival_date_month_num
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,0,Transient,0.00,0,0,Check-Out,2015-07-01,Portugal,6
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,0,Transient,0.00,0,0,Check-Out,2015-07-01,Portugal,6
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,0,Transient,75.00,0,0,Check-Out,2015-07-02,United Kingdom,6
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,0,Transient,75.00,0,0,Check-Out,2015-07-02,United Kingdom,6
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,0,Transient,98.00,0,1,Check-Out,2015-07-03,United Kingdom,6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
118893,City Hotel,0,23,2017,August,35,30,2,5,2,...,No Deposit,0,Transient,96.14,0,0,Check-Out,2017-09-06,Belgium,7
118894,City Hotel,0,102,2017,August,35,31,2,5,3,...,No Deposit,0,Transient,225.43,0,2,Check-Out,2017-09-07,France,7
118895,City Hotel,0,34,2017,August,35,31,2,5,2,...,No Deposit,0,Transient,157.71,0,4,Check-Out,2017-09-07,Germany,7
118896,City Hotel,0,109,2017,August,35,31,2,5,2,...,No Deposit,0,Transient,104.40,0,0,Check-Out,2017-09-07,United Kingdom,7


In [57]:
# Линейные модели logistic_regression / linear, Даниэль
#  svm Игорь
#  boosting Игорь
#  random trees Даниэль
#  нейронка Игорь
#  изотоническая регрессия Даниэль
#  ранжирование попробовать? Игорь

#  Для каждой затюнить гиперпараметры через GridSearchCV или optuna
#  Результаты по проведенным экспериментам собрать в pandas датасет и отправить
#  Сделать выводы по performance моделю, посмотреть на важность признаков через shap или feature_importances


#  Графики с дубликатами и без сравнить

In [58]:
df_model = df[[c for c in df.columns if c not in ['reservation_status', 'reservation_status_date', 'country_full']]]

In [59]:
df_model

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,reserved_room_type,assigned_room_type,booking_changes,deposit_type,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,arrival_date_month_num
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,C,C,3,No Deposit,0,Transient,0.00,0,0,6
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,C,C,4,No Deposit,0,Transient,0.00,0,0,6
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,A,C,0,No Deposit,0,Transient,75.00,0,0,6
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,A,A,0,No Deposit,0,Transient,75.00,0,0,6
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,A,A,0,No Deposit,0,Transient,98.00,0,1,6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
118893,City Hotel,0,23,2017,August,35,30,2,5,2,...,A,A,0,No Deposit,0,Transient,96.14,0,0,7
118894,City Hotel,0,102,2017,August,35,31,2,5,3,...,E,E,0,No Deposit,0,Transient,225.43,0,2,7
118895,City Hotel,0,34,2017,August,35,31,2,5,2,...,D,D,0,No Deposit,0,Transient,157.71,0,4,7
118896,City Hotel,0,109,2017,August,35,31,2,5,2,...,A,A,0,No Deposit,0,Transient,104.40,0,0,7


In [60]:
X = df_model[[c for c in df_model.columns if c != 'adr']].copy()
y = df_model.adr.copy()

In [62]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=8)

## SVM

In [92]:
num_features = [
    'lead_time', 'arrival_date_year', 'arrival_date_week_number',
    'arrival_date_day_of_month', 'stays_in_weekend_nights',
    'stays_in_week_nights', 'adults', 'children', 'babies',
    'is_repeated_guest', 'previous_cancellations',
    'previous_bookings_not_canceled', 'booking_changes',
    'days_in_waiting_list', 'required_car_parking_spaces',
    'total_of_special_requests'
]

cat_features = [
    'hotel', 'arrival_date_month', 'meal', 'country',
    'market_segment', 'distribution_channel', 'reserved_room_type',
    'assigned_room_type', 'deposit_type', 'customer_type'
]

In [93]:
preprocessor = ColumnTransformer(
    transformers=[
        ('scaler', StandardScaler(), num_features),
        ('cat', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), cat_features)
    ]
)

In [94]:
svm_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', LinearSVR( random_state=8)),
])

In [97]:
def objective(trial):

    max_iter_param = trial.suggest_int('regressor__max_iter', 1000, 10000, step=1000)
    tol_param = trial.suggest_float('regressor__tol', 1e-5, 1e-2, log=True)
    c_param = trial.suggest_float('regressor__C', 1e-4, 1e2, log=True)

    svm_pipeline.set_params(
        regressor__C=c_param,
        regressor__max_iter=max_iter_param,
        regressor__tol=tol_param
    )


    cv = KFold(n_splits=5, shuffle=True, random_state=8)

    scores = cross_val_score(
        svm_pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring='neg_mean_absolute_error',
        n_jobs=-1,
        error_score='raise'
    )

    return np.mean(scores)

In [98]:
study = optuna.create_study(direction='maximize')
study.optimize(objective, show_progress_bar=True, n_trials=10)

[I 2026-06-16 19:37:51,749] A new study created in memory with name: no-name-202a05d0-aef2-4764-871b-8b47cb0a0f6f


  0%|          | 0/10 [00:00<?, ?it/s]

[I 2026-06-16 19:37:58,430] Trial 0 finished with value: -22.253472339686347 and parameters: {'regressor__max_iter': 8000, 'regressor__tol': 0.00021260897094455258, 'regressor__C': 0.26350117813004725}. Best is trial 0 with value: -22.253472339686347.
[I 2026-06-16 19:37:59,716] Trial 1 finished with value: -22.985212365337812 and parameters: {'regressor__max_iter': 9000, 'regressor__tol': 0.0061713378143223455, 'regressor__C': 0.04914189417397742}. Best is trial 0 with value: -22.253472339686347.
[I 2026-06-16 19:38:01,194] Trial 2 finished with value: -24.153725533037512 and parameters: {'regressor__max_iter': 1000, 'regressor__tol': 2.653959104351434e-05, 'regressor__C': 0.016068913324146106}. Best is trial 0 with value: -22.253472339686347.
[I 2026-06-16 19:38:02,448] Trial 3 finished with value: -24.78365415804842 and parameters: {'regressor__max_iter': 4000, 'regressor__tol': 0.0032177430938603187, 'regressor__C': 0.011313190611892005}. Best is trial 0 with value: -22.25347233968

In [99]:
study.trials_dataframe().to_csv('svmreg_results.csv')

In [100]:
best = study.best_params

In [102]:
# svm_pipeline.set_params(
#     regressor__C=best['regressor__C'],
#     regressor__max_iter=best['regressor__max_iter'],
#     regressor__tol=best['regressor__tol']
# )

In [103]:
# import importlib
# import ml_utils as mu

# # Эта команда принудительно обновит модуль в памяти Jupyter
# importlib.reload(mu)

In [104]:
# mu.run_experiment(svm_pipeline, 'svm', 'main_df_edited_no_dup.parquet', best)